In [ ]:
# =============================================================================
# 03_trends_mann_kendall.ipynb  —  Rigorous Trend Analysis
# Standard Mann-Kendall + Sen's Slope.
# Continuous monthly interpolation for basin series.
# Block-level trend maps.
# =============================================================================

import sys, os, gc, time, pickle, warnings
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pymannkendall as mk

# --- Set Root Dir ---
REPO_ROOT = Path('/scratch/lustre/users/johngachoka/SOIL-MOISTURE-PREDICTION')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    PROCESSED_DIR, DERIVED_DIR, TRENDS_DIR, LOG_DIR, FIG_DIR,
    SEASONS, ALPHA, N_MIN_MK
)

warnings.filterwarnings('ignore')

# --- Config ---
N_WORKERS = 16
CACHE_TABLE = TRENDS_DIR / 'mk_cache_table.pkl'
CACHE_SERIES = TRENDS_DIR / 'mk_cache_series.pkl'
CACHE_MAPS = TRENDS_DIR / 'mk_cache_maps.npz'

# Ensure output dirs exist
TRENDS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

VARIABLES = {
    'NDVI':           (PROCESSED_DIR / 'NDVI_stack_qa.nc',        'NDVI'),
    'FVC':            (DERIVED_DIR   / 'FVC_stack.nc',            'FVC'),
    'FOREST_DENSITY': (DERIVED_DIR   / 'FOREST_DENSITY_stack.nc', 'FOREST_DENSITY'),
    'SM_L2':          (PROCESSED_DIR / 'SM_L2_stack.nc',          'SM_L2'),
    'SPEI_3':         (DERIVED_DIR   / 'SPEI_3_stack.nc',         'SPEI_3'),
    'PRECIP':         (PROCESSED_DIR / 'PRECIP_stack.nc',         'PRECIP'),
}

VAR_DISPLAY = {
    'NDVI':           ('ndvi',              'unitless'),
    'FVC':            ('fvc',               '0-1'),
    'FOREST_DENSITY': ('forest_density',    '0-1'),
    'SM_L2':          ('sm_l2',             'm³/m³'),
    'SPEI_3':         ('spei3',             'std'),
    'PRECIP':         ('precip',            'mm/season'),
}

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"Variables   : {list(VARIABLES.keys())}")
print(f"Cache exists: {CACHE_TABLE.exists() and CACHE_MAPS.exists()}")

In [ ]:
# =============================================================================
# Computes:
#   1. Basin-level continuous series (Interpolated to monthly) + MK
#   2. Pixel-wise Sen's slope + MK p-value for map generation
# Caches everything to disk.
# =============================================================================

def _mk_one_pixel(series, n_min):
    valid = series[np.isfinite(series)]
    if valid.size < n_min: return (np.nan, np.nan)
    if np.all(valid == valid[0]): return (0.0, 1.0)
    try:
        res = mk.original_test(valid)
        return (float(res.slope), float(res.p))
    except Exception:
        return (np.nan, np.nan)

_G_SERIES = None
_G_NMIN = None

def _init_worker(series_flat, n_min):
    global _G_SERIES, _G_NMIN
    _G_SERIES, _G_NMIN = series_flat, n_min

def _process_row(j):
    nx = _G_SERIES.shape[2]
    slope = np.full(nx, np.nan, dtype='float32')
    pval  = np.full(nx, np.nan, dtype='float32')
    for k in range(nx):
        s, p = _mk_one_pixel(_G_SERIES[:, j, k], _G_NMIN)
        slope[k], pval[k] = s, p
    return slope, pval

def run_parallel_maps(da, n_min=N_MIN_MK, n_workers=N_WORKERS):
    nt, ny, nx = da.shape
    series_flat = da.values.astype('float32')
    slope = np.full((ny, nx), np.nan, dtype='float32')
    pval  = np.full((ny, nx), np.nan, dtype='float32')
    with Pool(processes=n_workers, initializer=_init_worker,
              initargs=(series_flat, n_min)) as pool:
        for j, (s_row, p_row) in enumerate(tqdm(pool.imap(_process_row, range(ny), chunksize=16), total=ny, desc='Map Rows')):
            slope[j] = s_row; pval[j] = p_row
    return slope, pval

# =============================================================================
# 1. LOAD CACHE IF EXISTS
# =============================================================================
if CACHE_TABLE.exists() and CACHE_MAPS.exists():
    print("✓ Loading cached MK results and maps...")
    with open(CACHE_TABLE, 'rb') as f:
        table_monthly = pickle.load(f)
    with open(CACHE_SERIES, 'rb') as f:
        ts_store = pickle.load(f)
    maps_data = np.load(CACHE_MAPS, allow_pickle=True)
    map_slope = {tuple(k.split('_')): v for k, v in zip(maps_data['keys'], maps_data['slope'])}
    map_pval = {tuple(k.split('_')): v for k, v in zip(maps_data['keys'], maps_data['pval'])}
    
else:
    print("No cache → running heavy computation...")
    table_rows = []
    ts_store = {}
    map_slope = {}
    map_pval = {}
    
    for var_name, (path, var_key) in VARIABLES.items():
        print(f"\n▶ {var_name}")
        ds = xr.open_dataset(path)
        if var_key not in ds.data_vars:
            ds.close(); continue
        da_full = ds[var_key]

        # --- 1. Block-level Maps (Pixel-wise MK per season) ---
        for season in SEASONS:
            mask = (da_full.season.values == season)
            if not mask.any(): continue
            da_season = da_full.isel(time=mask)
            map_slope[(var_name, season)], map_pval[(var_name, season)] = run_parallel_maps(da_season)

        # --- 2. Basin Series (Continuous Monthly Anomalies) ---
        # Build raw seasonal series first
        xs_raw, ys_raw = [], []
        for season_idx, season in enumerate(SEASONS):
            mask = (da_full.season.values == season)
            if not mask.any(): continue
            da_season = da_full.isel(time=mask)
            regional_ts = np.nanmean(da_season.values.reshape(da_season.shape[0], -1), axis=1)
            
            for y, v in zip(da_season.year.values, regional_ts):
                xs_raw.append(float(y) + (season_idx * 0.25))
                ys_raw.append(v)
        
        # Sort and interpolate to smooth monthly (12 points/year)
        idx = np.argsort(xs_raw)
        xs_raw, ys_raw = np.array(xs_raw)[idx], np.array(ys_raw)[idx]
        
        # Create dense monthly time axis
        years = np.arange(np.floor(xs_raw.min()), np.ceil(xs_raw.max()) + 1)
        months = np.arange(1, 13)
        xs_dense = np.array([y + (m-1)/12 for y in years for m in months])
        
        # Interpolate (linear or cubic for smoothness)
        ys_dense = np.interp(xs_dense, xs_raw, ys_raw)
        
        ts_store[var_name] = (xs_dense, ys_dense)

        # --- 3. Mann-Kendall Stats on Continuous Series ---
        clean_ys = ys_dense[np.isfinite(ys_dense)]
        
        mk_res = mk.original_test(clean_ys)
        
        # Slope is per month step, so multiply by 12 for per-year
        slope_per_year = mk_res.slope * 12 
        
        table_rows.append({
            'variable': var_name,
            'n': len(clean_ys),
            'sen_per_year': slope_per_year,
            'mk_p': mk_res.p,
            'trend': mk_res.trend
        })

    table_monthly = pd.DataFrame(table_rows)
    
    # Save caches
    with open(CACHE_TABLE, 'wb') as f: pickle.dump(table_monthly, f)
    with open(CACHE_SERIES, 'wb') as f: pickle.dump(ts_store, f)
    
    keys_arr = np.array([f"{k[0]}_{k[1]}" for k in map_slope], dtype=object)
    np.savez_compressed(CACHE_MAPS, 
                        slope=np.array([map_slope[k] for k in map_slope]),
                        pval=np.array([map_pval[k] for k in map_pval]),
                        keys=keys_arr)

# Display Table
print("\n" + "="*80)
print("Table 1: Main Trend Summary (Continuous Monthly MK)")
print(table_monthly.round(4).to_string(index=False))
table_monthly.to_csv(LOG_DIR / "trend_table_main.csv", index=False)

In [ ]:
# =============================================================================
# Table 2: Driver vs Target Roles (Corrected for this project)
# Table 3: Period Sensitivity (1995-2012 vs 2013-2025)
# =============================================================================

# 1. Correct Roles based on YOUR project objectives
ROLES = {
    'SM_L2': 'target',           # Subsurface soil moisture is the target
    'FOREST_DENSITY': 'driver',  # Forest density is the main driver
    'NDVI': 'driver',
    'FVC': 'driver',
    'SPEI_3': 'driver',
    'PRECIP': 'driver'
}

t2 = table_monthly.copy()
t2['role'] = t2['variable'].map(ROLES)

# Format the table to match the clean PDF layout
t2_formatted = t2[['variable', 'n', 'sen_per_year', 'mk_p', 'role']].copy()
t2_formatted.columns = ['variable', 'n', 'sen_per_year', 'p', 'role']

print("\n--- Table 2: Main Trends & Roles ---")
print(t2_formatted.round(4).to_string(index=False))
t2_formatted.to_csv(LOG_DIR / 'table_roles.csv', index=False)

# 2. Period Sensitivity Table
rows = []
for var_name in VARIABLES.keys():
    if var_name not in ts_store: continue
    xs, ys = ts_store[var_name]
    
    versions = {
        'all months': ys,
        '1995-2012': np.where(xs < 2013, ys, np.nan),
        '2013-2025': np.where(xs >= 2013, ys, np.nan)
    }
    
    for label, data in versions.items():
        clean = data[np.isfinite(data)]
        if len(clean) > 10:
            res = mk.original_test(clean)
            rows.append({
                'variable': var_name,
                'version': label,
                'n': len(clean),
                'sen_per_year': res.slope * 12,
                'p': res.p
            })

t3 = pd.DataFrame(rows)
print("\n--- Table 3: Period Sensitivity ---")
print(t3.round(4).to_string(index=False))
t3.to_csv(LOG_DIR / 'table_sensitivity.csv', index=False)

In [ ]:
# =============================================================================
# Cell 4 — Presentation Outputs (Max 3 Figures)
# 1. Continuous Trend Graph (3 variables)
# 2. SM_L2 Drying Map (Slope + Significance)
# 3. Forest Density Map (Slope + Significance)
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import xarray as xr

# Variables to plot in the line graph
TREND_VARS = ['FOREST_DENSITY', 'SM_L2', 'SPEI_3']
COLORS = {'FOREST_DENSITY': '#2ca02c', 'SM_L2': '#17becf', 'SPEI_3': '#9467bd'}

# -----------------------------------------------------------------------------
# FIGURE 1: Continuous Trend Graph
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, var_name in zip(axes, TREND_VARS):
    if var_name not in ts_store: continue
    xs, ys = ts_store[var_name]
    
    # Recalculate slope for the plot line
    import pymannkendall as mk
    clean_ys = ys[np.isfinite(ys)]
    res = mk.original_test(clean_ys)
    trend_line = res.slope * xs + (np.nanmean(ys) - res.slope * np.nanmean(xs))
    
    ax.plot(xs, ys, color=COLORS[var_name], lw=1.2, label='Continuous Profile')
    ax.plot(xs, trend_line, '--', color='black', lw=2, 
            label=f"Sen Slope: {res.slope*12:.3g}/yr (p={res.p:.3f})")
    
    ax.set_ylabel(var_name.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)

axes[-1].set_xlabel('Year', fontsize=12)
plt.suptitle('Basin-Level Trajectories (Continuous Interpolated)', fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.savefig(FIG_DIR / 'pres_fig1_trends.png', dpi=150, bbox_inches='tight')
plt.show()


# -----------------------------------------------------------------------------
# MAPS: Spatial Slope & Significance (For a specific season, e.g. JJAS)
# -----------------------------------------------------------------------------
TARGET_SEASON = 'JJAS' # We'll map the dry season

# Get coordinate extent
ds_ref = xr.open_dataset(PROCESSED_DIR / 'NDVI_stack_qa.nc')
extent = [ds_ref.x.min(), ds_ref.x.max(), ds_ref.y.min(), ds_ref.y.max()]

def plot_slope_map(var_name, season, filename):
    slope = map_slope.get((var_name, season))
    pval = map_pval.get((var_name, season))
    
    if slope is None:
        print(f"No map data for {var_name} {season}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: All slopes
    vmax = np.nanmax(np.abs(slope))
    im1 = axes[0].imshow(slope, cmap='RdBu_r', extent=extent, vmin=-vmax, vmax=vmax)
    axes[0].set_title(f'{var_name} Sen Slope ({season})', fontsize=12)
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    
    # Panel 2: Significant slopes only
    sig_mask = np.where(pval < 0.05, slope, np.nan)
    im2 = axes[1].imshow(sig_mask, cmap='RdBu_r', extent=extent, vmin=-vmax, vmax=vmax)
    axes[1].set_title(f'{var_name} Significant Trend (p < 0.05)', fontsize=12)
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    
    plt.suptitle(f'Spatial Trends: {var_name.replace("_", " ").title()}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()

# Map 2: SM_L2 (Drying)
plot_slope_map('SM_L2', TARGET_SEASON, 'pres_map2_sm_l2.png')

# Map 3: Forest Density
plot_slope_map('FOREST_DENSITY', TARGET_SEASON, 'pres_map3_forest_density.png')

print(f"\n✓ Exactly 3 presentation files generated in {FIG_DIR}")